In [11]:
import pandas as pd
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Load data
conn = sqlite3.connect("../Data/churn.db")
df = pd.read_sql_query("SELECT * FROM customers", conn)
conn.close()

# Reapply the TotalCharges fix
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [12]:
# customerID is just a label, not useful for prediction - drop it
df_model = df.drop(columns=['customerID'])

# Simplify "No internet service" / "No phone service" to just "No"
# These mean the same thing as "No" for modeling purposes
cols_to_simplify = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
                     'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in cols_to_simplify:
    df_model[col] = df_model[col].replace({'No internet service': 'No', 'No phone service': 'No'})

print("Simplified. Check unique values now:")
for col in cols_to_simplify:
    print(f"{col}: {df_model[col].unique()}")

Simplified. Check unique values now:
MultipleLines: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineSecurity: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup: <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [13]:
# Encode categorical columns to numbers
label_encoders = {}
categorical_cols = df_model.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

print("Encoding complete. Sample of encoded data:")
df_model.head()

Encoding complete. Sample of encoded data:


C:\Users\raman\AppData\Local\Temp\ipykernel_20332\3432785510.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_model.select_dtypes(include='object').columns


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,1,0,1,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,1,1,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,0,0,1,0,1,1,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [14]:
# Separate features (X) from target (y)
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

# Split into train (80%) and test (20%) sets
# stratify=y ensures both sets keep the same ~74/26 churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nChurn rate in training set:", y_train.mean().round(3))
print("Churn rate in test set:", y_test.mean().round(3))

Training set size: (5634, 19)
Test set size: (1409, 19)

Churn rate in training set: 0.265
Churn rate in test set: 0.265


In [15]:
# Train Logistic Regression model
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_log = log_model.predict(X_test)
y_pred_proba_log = log_model.predict_proba(X_test)[:, 1]  # probability of churn

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


c:\Users\raman\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:
print("=== Logistic Regression Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_log):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_log):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_log):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_log):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_log):.3f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nFull Classification Report:")
print(classification_report(y_test, y_pred_log))

=== Logistic Regression Performance ===
Accuracy:  0.800
Precision: 0.644
Recall:    0.551
F1 Score:  0.594
ROC-AUC:   0.841

Confusion Matrix:
[[921 114]
 [168 206]]

Full Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.64      0.55      0.59       374

    accuracy                           0.80      1409
   macro avg       0.74      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409



In [17]:
# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("=== Random Forest Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_rf):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_rf):.3f}")

=== Random Forest Performance ===
Accuracy:  0.800
Precision: 0.659
Recall:    0.511
F1 Score:  0.575
ROC-AUC:   0.838


In [18]:
# See which features matter most to the model
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(feature_importance.to_string(index=False))

         Feature  Coefficient
    PhoneService    -1.267189
        Contract    -0.723212
  OnlineSecurity    -0.617874
     TechSupport    -0.575635
PaperlessBilling     0.394107
    OnlineBackup    -0.345266
      Dependents    -0.239434
DeviceProtection    -0.217679
   SeniorCitizen     0.170983
   MultipleLines     0.134194
 StreamingMovies    -0.090005
   PaymentMethod     0.085968
     StreamingTV    -0.083795
          tenure    -0.051698
         Partner     0.036137
  MonthlyCharges     0.034136
 InternetService    -0.017520
          gender     0.013717
    TotalCharges     0.000234


In [19]:
from sklearn.preprocessing import StandardScaler

# Scale features so coefficients are comparable
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Retrain Logistic Regression on scaled data
log_model_scaled = LogisticRegression(max_iter=1000, random_state=42)
log_model_scaled.fit(X_train_scaled, y_train)

y_pred_scaled = log_model_scaled.predict(X_test_scaled)
y_pred_proba_scaled = log_model_scaled.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression (Scaled) Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_scaled):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_scaled):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_scaled):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba_scaled):.3f}")

# Now check feature importance properly with scaled coefficients
feature_importance_scaled = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_model_scaled.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nProperly scaled feature importance:")
print(feature_importance_scaled.to_string(index=False))

=== Logistic Regression (Scaled) Performance ===
Accuracy:  0.800
Precision: 0.644
Recall:    0.551
ROC-AUC:   0.841

Properly scaled feature importance:
         Feature  Coefficient
          tenure    -1.224247
  MonthlyCharges     1.036313
        Contract    -0.605705
    TotalCharges     0.487445
    PhoneService    -0.377818
  OnlineSecurity    -0.279794
     TechSupport    -0.261962
PaperlessBilling     0.196298
    OnlineBackup    -0.164241
      Dependents    -0.108918
DeviceProtection    -0.103481
   PaymentMethod     0.094263
   MultipleLines     0.066361
   SeniorCitizen     0.063330
 StreamingMovies    -0.043808
     StreamingTV    -0.040742
         Partner     0.019200
 InternetService    -0.011569
          gender     0.007725


In [20]:
import joblib

# Save the trained model, scaler, and label encoders for use in the Streamlit app
joblib.dump(log_model_scaled, "../app/churn_model.pkl")
joblib.dump(scaler, "../app/scaler.pkl")
joblib.dump(label_encoders, "../app/label_encoders.pkl")
joblib.dump(list(X.columns), "../app/feature_columns.pkl")

print("Model and preprocessing objects saved to app/ folder!")

Model and preprocessing objects saved to app/ folder!
